# Lab 13/14 — Decoding strategies, measured

Turn the lecture's decoding rules into **measurements**: the distribution a model emits, and what each
strategy does to it. Read the [lab brief](Lab13_14.md) first. You are marked on **relationships between
your own numbers** — the sampling parts are seeded from your roll number, so your numbers are yours.

**Before you run anything:** set your identity in the next cell. Fill every prediction and explanation cell. Run top to bottom,
then run the final export cell and submit the two files it names.

In [ ]:
ROLL_NUMBER = "202518053"      # <- your roll number, e.g. "202512345"
NAME        = "Falak Parmar"      # <- your name

# You may change PROMPT to a sentence of your own — a personal one makes your numbers more clearly yours.
PROMPT = "One ring to rule them all, one ring to find them, one ring to bring them all and in the darkness bind them." # - Gandalf
# "Not all those who wander are lost." — Bilbo Baggins
# “If there's one thing I've learned from you, Master, it's that following direct orders isn't always the best way to solve a problem.” — Ashoka Tano

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."

## Setup

In [3]:
import json, platform, sys, hashlib
from pathlib import Path
import torch, torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

MODEL = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id

SEED = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**31)   # your sampling seed
inputs = tokenizer(PROMPT, return_tensors="pt")
PLEN = inputs["input_ids"].shape[1]

def next_logits(prompt):
    ids = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        return model(**ids).logits[0, -1]

RESULTS = {}
print("model", MODEL, "| vocab", model.config.vocab_size, "| seed", SEED)

model gpt2 | vocab 50257 | seed 1021702246


---
## Part 1 — The distribution and greedy

📝 **Predict.** The model scores every token in a ~50k vocabulary. Roughly what fraction of the
probability mass do you expect the **top 50** tokens to hold? Run greedy decoding twice — will the two
outputs be identical? Will greedy's first token be the **argmax** of the distribution?

**Answer:**
- The top 50 tokens will hold roughly 90 - 95% of the probability mass.
- The two outputs should be identical.
- Yes, Greedy's first token will be the **argmax** of the distribution.

In [8]:
logits = next_logits(PROMPT)
probs = F.softmax(logits, dim=-1)
top50_mass = torch.topk(probs, 50).values.sum().item()

g1 = model.generate(**inputs, max_new_tokens=40, do_sample=False)[0, PLEN:].tolist()
g2 = model.generate(**inputs, max_new_tokens=40, do_sample=False)[0, PLEN:].tolist()

RESULTS["dist"] = dict(vocab_size=int(model.config.vocab_size), top50_mass=round(top50_mass, 4),
                       greedy_ids_run1=g1, greedy_ids_run2=g2,
                       greedy_first_id=int(g1[0]), argmax_id=int(torch.argmax(logits)))
print(f"top-50 mass            : {top50_mass:.1%}")
print(f"greedy runs identical  : {g1 == g2}")
print(f"greedy first == argmax : {g1[0] == int(torch.argmax(logits))}")
print("greedy text:", tokenizer.decode(g1))

top-50 mass            : 71.9%
greedy runs identical  : True
greedy first == argmax : True
greedy text: 

The Lord said to Moses, "I will make you a man of the Lord, and I will make you a man of the Lord. And I will make you a man of the Lord


📝 **Explain.** Why are two greedy runs identical while two sampling runs (Part 2) will not be? Why
must greedy's first token equal the argmax? What does the top-50 mass tell you about the **long tail**
that the truncation strategies in Part 3 exist to cut?

**Answer:**
- Two greedy runs are identical because greedy is deterministic - it always picks the token with the highest probability.
- Two sampling runs will not be identical because sampling introduces randomness - even with the same seed, different random numbers will be generated.
- Greedy's first token must equal the argmax because greedy always selects the token with the highest probability.
- The top-50 mass tells us how much probability mass is concentrated in the top 50 tokens. A low top-50 mass indicates a long tail, meaning many tokens have non-negligible probabilities.

---
## Part 2 — Temperature

📝 **Predict.** As temperature `T` rises, what happens to the probability of the single most likely
token? And to the **diversity** of sampled continuations (fraction of distinct tokens)? Predict the
*direction* of each change before you measure.

**Answer:**
- As temperature `T` rises, the probability of the single most likely token decreases.
- And the diversity of fraction of distinct tokens increases.
- The *direction* of each change is that the probability of the single most likely token decreases and the diversity of sampled continuations increases.

In [9]:
Ts = [0.5, 1.0, 2.0]
p_top = [F.softmax(logits / T, dim=-1).max().item() for T in Ts]     # same logits, reshaped

def distinct_ratio(T, n=16, k=20):
    toks = []
    for i in range(n):
        torch.manual_seed(SEED + i)                                 # seeded from your roll number
        out = model.generate(**inputs, max_new_tokens=k, do_sample=True,
                             temperature=T, top_k=0, top_p=1.0)
        toks += out[0, PLEN:].tolist()
    return len(set(toks)) / len(toks)

div_Ts = [0.7, 1.0, 1.5]
distinct = [round(distinct_ratio(T), 4) for T in div_Ts]

RESULTS["temperature"] = dict(Ts=Ts, p_top=[round(x, 4) for x in p_top],
                              div_Ts=div_Ts, distinct_ratio=distinct)
print("P(top token) at T =", Ts, "->", [f"{x:.2%}" for x in p_top])
print("distinct-token ratio at T =", div_Ts, "->", distinct)

P(top token) at T = [0.5, 1.0, 2.0] -> ['84.58%', '23.61%', '1.07%']
distinct-token ratio at T = [0.7, 1.0, 1.5] -> [0.45, 0.6875, 0.9299]


📝 **Explain.** Temperature divides the logits before softmax. Using the ratio `P_i / P_j = exp((l_i − l_j) / T)`, explain **why** raising `T` flattens the distribution and **why**
that raises diversity. What happens in the limit `T → 0`?

**Answer:**
- Raising T flattens the distribution because it reduces the relative difference, `(l_i - l_j)` between logits, making the softmax function output probabilities that are closer to each other, giving it a chance to choose multiple different tokens with similar probabiltes without one dominating other.
- This raises diversity because the model is more likely to sample from a wider range of tokens, rather than being locked into a few high-probability tokens.
- In the limit T → 0, the softmax function approaches a one-hot distribution, where the probability of the highest logit approaches 1 and all others approach 0.

---
## Part 3 — Truncation: top-k vs nucleus (top-p)

📝 **Predict.** On a **peaked** prompt (one obvious next word) versus a **flat** prompt (many options):
how many tokens will nucleus (top-p) keep in each? How many will top-k keep in each? Which one *adapts*?

**Answer:**
- On a peaked prompt, top-p will keep fewer tokens because the probability mass is concentrated in a smaller set of tokens. Top-k will keep a fixed number of tokens regardless of the prompt.
- On a flat prompt, top-p will keep more tokens because the probability mass is spread out over a larger set of tokens. Top-k will keep a fixed number of tokens regardless of the prompt.
- Top-p adapts because it dynamically adjusts the number of tokens based on the probability distribution, while top-k keeps a fixed number of tokens.

In [10]:
PEAKED = "The United States of"     # one obvious next token
FLAT   = "My favourite food is"     # many reasonable ones
P, K = 0.9, 50

def topp_stats(lg, p):
    s, _ = torch.sort(F.softmax(lg, dim=-1), descending=True)
    n = int((torch.cumsum(s, dim=-1) < p).sum()) + 1               # smallest set reaching p
    kept = s[:n].sum().item()
    kept_minus_last = s[:n-1].sum().item() if n > 1 else 0.0
    return n, kept, kept_minus_last

n_peak, _, _ = topp_stats(next_logits(PEAKED), P)
n_flat, kept_flat, kept_flat_minus1 = topp_stats(next_logits(FLAT), P)

RESULTS["truncation"] = dict(peaked_prompt=PEAKED, flat_prompt=FLAT, nucleus_p=P, topk_k=K,
                             nucleus_peaked=n_peak, nucleus_flat=n_flat,
                             topk_peaked=K, topk_flat=K,                     # top-k is fixed by definition
                             topp_kept_mass=round(kept_flat, 4),
                             topp_kept_minus_last=round(kept_flat_minus1, 4))
print(f"nucleus p={P}: peaked keeps {n_peak:>4} tokens | flat keeps {n_flat:>4} tokens")
print(f"top-k  k={K}: keeps {K} on both (fixed)")
print(f"flat nucleus kept mass = {kept_flat:.3f} (>= {P}?) ; without its last token = {kept_flat_minus1:.3f} (< {P}?)")

nucleus p=0.9: peaked keeps    1 tokens | flat keeps 1913 tokens
top-k  k=50: keeps 50 on both (fixed)
flat nucleus kept mass = 0.900 (>= 0.9?) ; without its last token = 0.900 (< 0.9?)


📝 **Explain.** Why does nucleus keep a different number of tokens on the two prompts while top-k
keeps the same number on both? Name the failure each one fixes — and the failure each one still has.

**Answer:**
- Top-p keeps a different number of tokens because it dynamically adjusts based on the probability distribution. On a peaked prompt, the probability mass is concentrated in a smaller set of tokens, so fewer tokens are needed to capture 90% of the mass. On a flat prompt, the probability mass is spread out over a larger set of tokens, so more tokens are needed.
- Top-k keeps the same number of tokens because it always selects the k topmost tokens regardless of the prompt.
- Top-p fixes the issue of fixed token selection by adapting to the probability distribution, while top-k still has the issue of fixed token selection.
- Top-p still has the issue of not being able to handle prompts with a large number of tokens that have similar probabilities.

---
## Part 4 — Beam vs greedy (the honest one)

Folklore says beam search, by keeping the `k` best partial sequences, finds a **higher-probability**
sequence than greedy. Measure whether it actually does on *your* run.

📝 **Predict.** Will beam's total sequence log-probability beat greedy's? Why might a *heuristic*
(non-exhaustive) search fail to?

**Answer:**
- Beam's total sequence log-probability will likely beat greedy's because beam search keeps multiple candidate sequences and explores a larger portion of the search space.
- A heuristic search might fail to find the globally optimal sequence because it only explores a limited number of candidate sequences and may miss better options.

In [11]:
def seq_logprob(full_ids):
    with torch.no_grad():
        lp = F.log_softmax(model(full_ids).logits[0, :-1], dim=-1)
    tgt = full_ids[0, 1:]
    return lp[PLEN-1:].gather(1, tgt[PLEN-1:].unsqueeze(1)).sum().item()   # log-prob of the generated tokens

greedy_out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
beam_out   = model.generate(**inputs, max_new_tokens=30, num_beams=5,
                            do_sample=False, length_penalty=0.0, early_stopping=False)
gl, bl = seq_logprob(greedy_out), seq_logprob(beam_out)

RESULTS["beam"] = dict(num_beams=5, greedy_logprob=round(gl, 3), beam_logprob=round(bl, 3),
                       beam_won=bool(bl >= gl))
print(f"greedy log-prob : {gl:.2f}")
print(f"beam   log-prob : {bl:.2f}")
print("beam won (>= greedy)?", bl >= gl)

greedy log-prob : -44.21
beam   log-prob : -35.91
beam won (>= greedy)? True


📝 **The paragraph that carries this part.** Did beam beat greedy on *your* run? Beam is not
exhaustive — it prunes low-scoring prefixes. Explain how greedy's path can be **dropped** from the beam
even though greedy would have recovered, and what that says about "higher probability = better output".
If beam *did* win, say what it found that greedy could not see.

**Answer:**
- Beam beat greedy on my run. Beam found a path that greedy could not see because beam explores multiple candidate sequences and keeps track of the best ones, while greedy only explores one path at a time. This shows that "higher probability = better output" is not always true because beam can find a better output by exploring multiple paths and keeping track of the best ones.


📝 **Closing — choose and justify.** For **code generation**, a **chatbot reply**, and **machine
translation** : name the decoding strategy you would serve each with, in one line each, and tie the
choice to a number you measured above.

**Answer:**
- Code generation: Top-p (nucleus) with p=0.9 because it allows for creativity while maintaining coherence.
- Chatbot reply: Beam search with beam size=4 because it provides a balance between speed and quality.
- Machine translation: Greedy decoding because it provides consistent and predictable responses. 

---
## Submit

Run this last. It writes `submission_lab13_14_<roll>.json`. Submit **two files**: that JSON and this
**executed notebook** with every prediction and explanation cell filled in.

In [12]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           torch=torch.__version__, transformers=transformers.__version__,
           model=MODEL, seed=SEED, prompt=PROMPT, linux=sys.platform.startswith("linux"))
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS)

out = Path(f"submission_lab13_14_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))
print("wrote", out)
print("  parts recorded:", list(RESULTS))
assert set(RESULTS) >= {"dist", "temperature", "truncation", "beam"}, "run every part before exporting"

wrote submission_lab13_14_202518053.json
  parts recorded: ['dist', 'temperature', 'truncation', 'beam']
